# S4 dev-small 청크 크기 비교

dev-small에서 고정 토큰 청킹 후보 네 개(256·512·1024·8192 토큰)를 비교해 베이스라인 청크 크기를 정한 과정을 기록한다.

- 임베딩은 로컬 PC에서 돌리기에 무거워 Colab L4에서 만들었다(`docs/slices/04-index.md` 실행 기록). 이 노트북은 그 인덱스를 불러와 확인하고, 검색과 채점을 다시 실행한다
- 결정 규칙은 점수를 보기 전에 지시서에 정했다: Recall@5 최고 후보를 기준으로, 차이의 95% 구간(회의 단위 bootstrap)이 0을 포함하는 후보 중 청크 수가 가장 적은 것을 고른다
- 결과와 결정은 `docs/DECISIONS.md` D-04에 있다

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from transformers import AutoTokenizer

from rag.chunk_sweep import query_metrics
from rag.eval.bootstrap import paired_bootstrap
from rag.eval.metrics import gold_ranks, load_gold, min_docs, read_run
from rag.index import load_cfg, load_embeddings
from rag.search import rank_docs

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
LAYER = "dev-small"
PRIMARY = "recall@5"

cfg = load_cfg()
candidates = [tuple(c) for c in cfg.chunk_sweep.candidates]
recall_ks, mrr_ks = list(cfg.chunk_sweep.recall_ks), list(cfg.chunk_sweep.mrr_ks)
bs = cfg.eval.bootstrap

pd.set_option("display.max_colwidth", 80)
candidates

[(256, 32), (512, 64), (1024, 128), (8192, 1024)]

In [3]:
def index_dir(ct: int, ov: int) -> Path:
    """후보 설정의 dev-small 인덱스 경로. config의 paths.index_dir 규칙을 그대로 쓴다."""
    c = load_cfg([f"chunking.chunk_tokens={ct}", f"chunking.overlap_tokens={ov}"])
    return ROOT / c.paths.index_dir


def run_path(ct: int, ov: int) -> Path:
    """rag.search가 쓴 순위 파일 경로."""
    return ROOT / cfg.paths.runs_dir / f"{index_dir(ct, ov).parent.name}-{LAYER}.csv"

## 1. 데이터: dev-small 질의와 문서

평가 층 dev-small은 dev 회의 일부의 전체 문서(약 1,000건)와, 그 문서가 정답인 dev 질의다. 질의마다 정답 문서는 하나다.

In [4]:
gold = load_gold(cfg, LAYER)
print(gold.shape)
gold.head()

(1039, 4)


,doc_id,qna_type,conference_number,meeting_name
qid,,,,
021857-0001,021857-2b9fce6e51,추출형,021857,예산결산특별위원회
021890-0001,021890-6f724b08e1,추출형,021890,예산결산특별위원회
028740-0001,028740-772e6c1a9b,추출형,028740,특별위원회
028866-0001,028866-60cdb78813,추출형,028866,본회의
028893-0001,028893-64046f2975,추출형,028893,국정감사


In [5]:
pd.concat(
    [gold["meeting_name"].value_counts(), gold["qna_type"].value_counts()],
    keys=["회의구분", "qna_type"],
).to_frame("질의 수")

질의 수
회의구분     소위원회        438
         국정감사        345
         특별위원회       105
         본회의          95
         예산결산특별위원회    56
qna_type 추출형         941
         단답형          98

문서 길이를 KURE-v1 토크나이저 기준 토큰 수로 본다. 청크 크기 후보를 이 분포와 견주면 크기마다 몇 개 문서가 잘리는지 짐작할 수 있다.

In [6]:
small_docs = set((ROOT / cfg.paths.dev_small_docs).read_text(encoding="utf-8").split())
with (ROOT / cfg.paths.corpus).open(encoding="utf-8") as f:
    docs = {d["doc_id"]: d["context"] for d in map(json.loads, f) if d["doc_id"] in small_docs}
print(len(docs), "docs")

1007 docs


In [7]:
tokenizer = AutoTokenizer.from_pretrained(cfg.embedding.model_name)
doc_tokens = pd.Series(
    {
        doc_id: len(tokenizer(text, add_special_tokens=False)["input_ids"])
        for doc_id, text in docs.items()
    }
)
doc_tokens.describe(percentiles=[0.5, 0.9, 0.99]).round(0)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (14265 > 8192). Running this sequence through the model will result in indexing errors


count     1007.0
mean      1164.0
std       1164.0
min         14.0
50%        929.0
90%       2218.0
99%       5407.0
max      14265.0
dtype: float64

In [8]:
pd.DataFrame(
    {"잘리는 문서 비율(%)": [round(100 * (doc_tokens > ct).mean(), 1) for ct, _ in candidates]},
    index=[ct for ct, _ in candidates],
).rename_axis("chunk_tokens")

,잘리는 문서 비율(%)
chunk_tokens,
256,85.1
512,70.1
1024,44.8
8192,0.4


## 2. 청킹 결과

청크는 토큰 `[i·s, i·s + chunk_tokens)` 구간이고 `s = chunk_tokens − overlap_tokens`다. overlap은 크기의 1/8이다. 청크 텍스트는 토큰 offset으로 원문을 잘라 얻는다(`src/rag/chunking.py`).

In [9]:
chunks = {
    ct: pd.read_json(index_dir(ct, ov) / "chunks.jsonl", lines=True)
    for ct, ov in candidates
}
pd.DataFrame.from_dict(
    {
        ct: {
            "overlap": ov,
            "청크 수": len(c),
            "문서 수": c["doc_id"].nunique(),
            "문서당 청크": round(len(c) / c["doc_id"].nunique(), 2),
            "평균 토큰": round(c["n_tokens"].mean(), 1),
            "최대 토큰": c["n_tokens"].max(),
        }
        for (ct, ov), c in zip(candidates, chunks.values(), strict=True)
    },
    orient="index",
).rename_axis("chunk_tokens")

,overlap,청크 수,문서 수,문서당 청크,평균 토큰,최대 토큰
chunk_tokens,,,,,,
256,32,5590,1007,5.55,235.9,256
512,64,2995,1007,2.97,433.8,512
1024,128,1747,1007,1.73,725.1,1024
8192,1024,1011,1007,1.00,1163.4,8192


512 후보에서 청크 세 개로 나뉜 문서 하나를 골라 경계를 본다. 이웃 청크는 64토큰씩 겹친다.

In [10]:
c512 = chunks[512]
sample_doc = c512.groupby("doc_id").size().loc[lambda s: s == 3].index[0]
sample = c512[c512["doc_id"] == sample_doc]
sample.assign(
    앞부분=sample["text"].str[:60],
    뒷부분=sample["text"].str[-60:],
)[["chunk_id", "start", "end", "n_tokens", "앞부분", "뒷부분"]]

,chunk_id,start,end,n_tokens,앞부분,뒷부분
17,17,0,1047,512,잘 들었습니다. 다음은 행정자치부장관께 질문하겠습니다. 모두질문에 대한 행자부장관의 답변을 잘 들었습,행자부 입장에서는 지금 현재 추진되고 있는 전남도청…… 입장이 아니라 내가 질문을 그렇게 했잖아요. 이것은
18,18,919,2003,512,"언반구도 없어요. 왜 그런 식으로 답변을 합니까? 내가 질문한 것, 메모도 안 해요! 그리고 대통령께 진언하","? 연구해 보지 않으셨지요? 그리고 지금 가장 중요한 것이 행정구역 개편입니다, 그렇지요? 예. 행정구역"
19,19,1873,2938,512,의한 인물 발탁은 할 수 없게 될 것입니다. 사실 그렇지요? 전자정부 하에서 행정고시를 통한 인재를 발굴해서,려져 있습니다. 그런 말을 한 적이 전혀 없습니다. 시간이 다 되어서 다음에 더 묻겠습니다. 감사합니다.


In [11]:
first, second = sample.iloc[0], sample.iloc[1]
overlap_text = first["text"][second["start"] - first["start"]:]
print(f"겹치는 구간: 문자 {second['start']}~{first['end']} ({len(overlap_text)}자)")
print(overlap_text)

겹치는 구간: 문자 919~1047 (128자)
언반구도 없어요. 왜 그런 식으로 답변을 합니까? 내가 질문한 것, 메모도 안 해요! 그리고 대통령께 진언하겠습니까? 저희 행자부 입장에서는 지금 현재 추진되고 있는 전남도청…… 입장이 아니라 내가 질문을 그렇게 했잖아요. 이것은


## 3. 인덱스 확인

Colab에서 만든 인덱스가 문서를 빠짐없이 담고, 청크 행 수와 임베딩 행 수가 맞는지 본다. 벡터는 정규화해 저장했으므로 노름이 1에 가까워야 한다(float16으로 계산해 오차가 남는다).

In [12]:
rows = {}
for ct, ov in candidates:
    d = index_dir(ct, ov)
    meta = json.loads((d / "meta.json").read_text(encoding="utf-8"))
    emb = load_embeddings(d)
    rows[ct] = {
        "임베딩 행": emb.shape[0],
        "청크 행": len(chunks[ct]),
        "문서 전부 포함": set(chunks[ct]["doc_id"]) == small_docs,
        "NaN": bool(np.isnan(emb).any()),
        "노름 최대 오차": round(float(np.abs(np.linalg.norm(emb, axis=1) - 1).max()), 4),
        "장치": f"{meta['device']}/{meta['dtype']}",
        "임베딩 초": meta["seconds"],
    }
pd.DataFrame(rows).T.rename_axis("chunk_tokens")

,임베딩 행,청크 행,문서 전부 포함,NaN,노름 최대 오차,장치,임베딩 초
chunk_tokens,,,,,,,
256,5590,5590,True,False,0.0005,cuda/float16,74.1
512,2995,2995,True,False,0.0005,cuda/float16,46.8
1024,1747,1747,True,False,0.0005,cuda/float16,48.0
8192,1011,1011,True,False,0.0005,cuda/float16,55.0


## 4. 검색

질의 임베딩은 로컬 CPU(float32)에서 한 번 만들어 둔 캐시를 쓴다. 평가 질의는 Colab에 올리지 않았다. 검색은 모든 청크와 내적을 계산하는 전수 검색(exact kNN)이고, 청크 순위에서 문서가 처음 나온 순위를 문서 순위로 쓴다.

In [13]:
cached = np.load(ROOT / cfg.paths.query_emb)
query_emb = cached["emb"]
assert cached["qids"].tolist() == gold.index.tolist()
query_emb.shape

(1039, 1024)

질의 하나로 512 후보의 상위 5개 문서를 본다.

In [14]:
qid = gold.index[0]
with (ROOT / cfg.paths.queries).open(encoding="utf-8") as f:
    query_text = next(q["query"] for q in map(json.loads, f) if q["qid"] == qid)
print(qid, query_text)
print("정답:", gold.loc[qid, "doc_id"])

emb512 = load_embeddings(index_dir(512, 64))
top5 = rank_docs(query_emb[gold.index.get_loc(qid)] @ emb512.T, c512["doc_id"].to_numpy(), 5)
pd.DataFrame(
    {"doc_id": top5, "정답": [d == gold.loc[qid, "doc_id"] for d in top5]}, index=range(1, 6)
)

021857-0001 한․일관계에서 우리나라 외무부가 지켜야 할 가장 기본적인 과제는 무엇이라고 생각하십니까?
정답: 021857-2b9fce6e51


,doc_id,정답
1,021857-2b9fce6e51,True
2,049487-41cfe2bd95,False
3,032739-06252d89e5,False
4,050738-f757d253e7,False
5,051488-c679178c0d,False


네 후보 모두 검색해 순위를 만들고, `rag.search` 명령이 저장한 순위 파일과 같은지 확인한다.

In [15]:
runs = {}
for ct, ov in candidates:
    scores = query_emb @ load_embeddings(index_dir(ct, ov)).T
    doc_ids = chunks[ct]["doc_id"].to_numpy()
    runs[ct] = pd.DataFrame(
        [
            {"qid": q, "rank": r, "doc_id": d}
            for q, row in zip(gold.index, scores, strict=True)
            for r, d in enumerate(rank_docs(row, doc_ids, cfg.retriever.top_k), 1)
        ]
    )
{ct: runs[ct].equals(read_run(run_path(ct, ov))) for ct, ov in candidates}

{256: True, 512: True, 1024: True, 8192: True}

## 5. 평가

주지표는 Recall@5, 보조 지표는 Recall@1·5·10과 MRR@1·5·10이다. 질의마다 서로 다른 문서가 10개 이상인지 평가 모듈이 검증한다.

In [16]:
n_min = min_docs(recall_ks, max(mrr_ks))
scores = {
    ct: query_metrics(gold_ranks(runs[ct], gold, n_min), recall_ks, mrr_ks)
    for ct, _ in candidates
}
overall = pd.DataFrame({ct: s.mean() for ct, s in scores.items()}).T.round(4)
overall.insert(0, "청크 수", [len(chunks[ct]) for ct in overall.index])
overall.rename_axis("chunk_tokens")

,청크 수,recall@1,recall@5,recall@10,mrr@1,mrr@5,mrr@10
chunk_tokens,,,,,,,
256,5590,0.8845,0.9721,0.9827,0.8845,0.9209,0.9223
512,2995,0.8739,0.9673,0.9817,0.8739,0.9134,0.9151
1024,1747,0.8431,0.9586,0.9788,0.8431,0.8915,0.8940
8192,1011,0.8056,0.9423,0.9673,0.8056,0.8641,0.8675


Recall@5가 가장 높은 후보를 기준으로 다른 후보와의 차이를 회의 단위 paired bootstrap(10,000회)으로 구한다. 같은 회의의 질의는 서로 독립이 아니어서 회의 단위로 재표본한다. 단위는 %p다.

In [17]:
best = overall[PRIMARY].idxmax()
groups = gold[bs.unit].to_numpy()


def diff_vs_best(ct, metric):
    """후보 ct와 기준 후보의 질의별 차이의 (점 추정, 95% 하한, 상한)."""
    d = (scores[ct][metric] - scores[best][metric]).to_numpy()
    return paired_bootstrap(d, groups, bs.n_resamples, bs.seed, bs.alpha)


diff_rows = {}
for ct, _ in candidates:
    ci = {m: diff_vs_best(ct, m) for m in [PRIMARY, "recall@1", "mrr@10"]}
    diff_rows[ct] = {
        m: f"{100 * p:+.2f} [{100 * lo:+.2f}, {100 * hi:+.2f}]" for m, (p, lo, hi) in ci.items()
    }
    diff_rows[ct]["R@5 구간이 0 포함"] = ci[PRIMARY][1] <= 0 <= ci[PRIMARY][2]
print("기준 후보:", best)
pd.DataFrame(diff_rows).T.rename_axis("chunk_tokens")

기준 후보: 256


,recall@5,recall@1,mrr@10,R@5 구간이 0 포함
chunk_tokens,,,,
256,"+0.00 [+0.00, +0.00]","+0.00 [+0.00, +0.00]","+0.00 [+0.00, +0.00]",True
512,"-0.48 [-1.33, +0.34]","-1.06 [-2.70, +0.59]","-0.72 [-1.65, +0.22]",True
1024,"-1.35 [-2.64, -0.10]","-4.14 [-6.56, -1.81]","-2.84 [-4.37, -1.30]",False
8192,"-2.98 [-4.57, -1.46]","-7.89 [-10.79, -5.10]","-5.48 [-7.42, -3.64]",False


회의구분·`qna_type`별 Recall@5다. 작은 청크의 이점이 어느 집단에서 나오는지 본다.

In [18]:
by_group = {
    col: pd.DataFrame(
        {ct: s[PRIMARY].groupby(gold[col]).mean() for ct, s in scores.items()}
    ).round(4)
    for col in cfg.eval.breakdowns
}
by_group["meeting_name"].assign(질의수=gold["meeting_name"].value_counts())

,256,512,1024,8192,질의수
meeting_name,,,,,
국정감사,0.9884,0.9797,0.9710,0.9478,345
본회의,0.9684,0.9579,0.9684,0.9263,95
소위원회,0.9612,0.9612,0.9521,0.9521,438
예산결산특별위원회,0.9286,0.9286,0.9107,0.9286,56
특별위원회,0.9905,0.9810,0.9619,0.9048,105


In [19]:
by_group["qna_type"].assign(질의수=gold["qna_type"].value_counts())

,256,512,1024,8192,질의수
qna_type,,,,,
단답형,0.9490,0.9490,0.9490,0.9490,98
추출형,0.9745,0.9692,0.9596,0.9416,941


256과 512가 Recall@1에서 서로 다르게 맞힌 질의 수다. 한쪽만 맞힌 질의가 양쪽에 고르게 있으면 두 크기가 서로 다른 질의에 강하다는 뜻이고, 여러 크기를 함께 쓰는 다중 크기 청킹(백로그 H9)의 근거가 된다.

In [20]:
hit1 = pd.DataFrame({ct: s["recall@1"] for ct, s in scores.items()}).astype(bool)
pd.crosstab(hit1[256], hit1[512], rownames=["256 R@1"], colnames=["512 R@1"])

512 R@1,False,True
256 R@1,,
False,91,29
True,40,879


## 6. 결정

- 규칙 적용: 기준 후보 256 대비 512의 Recall@5 차이 구간이 0을 포함하고 1024·8192는 포함하지 않는다. 0을 포함하는 후보(256, 512) 중 청크가 적은 512를 고른다
- 보조 지표(R@1, MRR@10)도 256이 근소하게 높지만 512와의 차이는 모두 잡음 범위다
- 사용자 판단 근거: 256 청크에는 답변에 필요한 내용이 충분히 담기지 않는다. 256은 벡터 수가 약 1.9배다. dev-full은 방해 문서가 약 38배라 작은 청크의 근소한 우위가 유지된다는 보장이 없다
- 결정: `chunk_tokens=512`, `overlap_tokens=64` (`configs/chunking/fixed.yaml`, DECISIONS D-04)
- 한계: dev-small은 방해 문서가 약 1,000개라 전체 코퍼스에서 순위가 달라질 수 있다. 전체 규모 확인은 백로그 H3에서 한다